In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

X_full = np.load('../data/X_train.npy')
y_full = np.load('../data/y_train.npy')
obs_count_full = np.load('../data/obs_count.npy')
obs_sse_full = np.load('../data/obs_sse.npy')

In [ ]:
n = X_full.shape[0]
n_train = (n * 4) // 5

np.random.seed(305)
shuffled_idx = np.random.permutation(range(n))
train_idx = shuffled_idx[:n_train]
test_idx = shuffled_idx[n_train:]

In [ ]:
np.save('X_train.npy', X_train:=X_full[train_idx])
np.save('y_train.npy', y_train:=y_full[train_idx])
np.save('obs_count.npy', obs_count_full[train_idx])
np.save('obs_sse.npy', obs_sse_full[train_idx])

np.save('X_test.npy', X_test:=X_full[test_idx])
np.save('y_test.npy', y_test:=y_full[test_idx])

In [ ]:
from itertools import product

ls_xy_vals  = [0.015, 0.03, 0.06, 0.12, 0.24]
ls_z_vals   = [0.25, 0.5, 1.0]
os_xyz_vals = [25, 75, 150]
ls_t_vals   = [0.02, 0.1, 0.5, 5.0, 50.0]
os_t_vals   = [5, 25, 75]
ls_ap_vals  = [0.5, 1.0, 2.0, 1e6]

grid = list(product(ls_xy_vals, ls_z_vals, os_xyz_vals, ls_t_vals, os_t_vals, ls_ap_vals))
param_grid = pd.DataFrame(grid, columns=["ls_xy", "ls_z", "os_xyz", "ls_t", "os_t", "ls_ap"])
param_grid.to_csv('param_grid3.csv', index=None)

In [ ]:
param_grid.shape

# KNN Baseline

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV

In [ ]:
X_geo_tr = np.load('../data/geo_X_train.npy')
y_geo_tr = np.load('../data/geo_y_train.npy')
X_geo_val = np.load('../data/geo_X_val.npy')
y_geo_val = np.load('../data/geo_y_val.npy')

In [ ]:
param_grid = {
    "n_neighbors": np.arange(1, 21),
    "weights": ["uniform", "distance"],
    "p": [1, 2]  # Manhattan vs Euclidean
}

knn_grid = GridSearchCV(
    KNeighborsRegressor(),
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",  # or "r2"
    n_jobs=-1,   # use all CPU cores
    verbose=0
)

knn_grid.fit(X_train, y_train)
print(f"n_neighbors: {knn_grid.best_estimator_.n_neighbors.item()}")

In [ ]:
knn_grid.best_score_

In [ ]:
preds = knn_grid.predict(X_test)
sq_errs = (preds - y_test)**2
mse = sq_errs.mean()

plt.hist(sq_errs)
plt.title(f"MSE: {mse:.4f}")
plt.show()

In [ ]:
CV_SPLIT_TYPES = ['rand', 'geo']
CV_N_SPLITS = 5
KNN_CV_RESULTS_PATH = 'knn_cv_mse_results.csv'


def fit_evaluate_knn_cv_split(split_type, split_idx):
    split_dir = f'../cv/{split_type}'

    X_cv_train = np.load(f'{split_dir}/X_train_{split_idx}.npy')
    y_cv_train = np.load(f'{split_dir}/y_train_{split_idx}.npy')
    X_cv_test = np.load(f'{split_dir}/X_test_{split_idx}.npy')
    y_cv_test = np.load(f'{split_dir}/y_test_{split_idx}.npy')

    knn_grid = GridSearchCV(
        KNeighborsRegressor(),
        param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=0,
    )
    knn_grid.fit(X_cv_train, y_cv_train)

    preds = knn_grid.predict(X_cv_test)
    mse = np.mean((preds - y_cv_test) ** 2)

    best_params = knn_grid.best_params_
    return {
        'split_type': split_type,
        'split_idx': split_idx,
        'n_train': X_cv_train.shape[0],
        'n_test': X_cv_test.shape[0],
        'mse': mse,
        'inner_cv_mse': -knn_grid.best_score_,
        'n_neighbors': best_params['n_neighbors'],
        'weights': best_params['weights'],
        'p': best_params['p'],
    }


knn_cv_results = []
for split_type in CV_SPLIT_TYPES:
    for split_idx in range(CV_N_SPLITS):
        print(f'Fitting KNN {split_type} split {split_idx}')
        result = fit_evaluate_knn_cv_split(split_type, split_idx)
        knn_cv_results.append(result)
        print(
            f"{split_type} split {split_idx}: "
            f"MSE={result['mse']:.4f}, "
            f"n_neighbors={result['n_neighbors']}, "
            f"weights={result['weights']}, p={result['p']}"
        )

knn_cv_results_df = pd.DataFrame(knn_cv_results)
knn_cv_results_df.to_csv(KNN_CV_RESULTS_PATH, index=False)
knn_cv_results_df